# Stage 2: Signal Processing & Feature Engineering
# Here, we apply the Hodrick-Prescott (HP) filter to separate long-term economic trends from short-term market static, ensuring the model trains on true systemic imbalances.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Load cleaned data from Notebook 1
df = pd.read_csv('data/01_cleaned_raw.csv')

# 2. Feature Engineering: Credit Velocity
df['credit_gdp'] = df['tloans'] / df['gdp']
df['credit_gdp_diff2'] = df.groupby('country')['credit_gdp'].diff(2)

# 3. Digital Filtering (HP Filter)
def apply_hp_filter(series, lamb=100):
    if series.dropna().empty:
        return series
    cycle, _ = sm.tsa.filters.hpfilter(series.dropna(), lamb=lamb)
    return cycle.reindex(series.index)

print("Applying HP Filters...")
df['credit_gdp_cycle'] = df.groupby('country')['credit_gdp'].transform(apply_hp_filter)
df['yield_curve_slope'] = df['ltrate'] - df['stir']
df['yield_curve_cycle'] = df.groupby('country')['yield_curve_slope'].transform(apply_hp_filter)

# 4. Define final features and impute missing values
features = ['yield_curve_slope', 'credit_gdp', 'credit_gdp_diff2', 'credit_gdp_cycle', 'yield_curve_cycle', 'cpi', 'unemp', 'debtgdp']
for col in features:
    df[col] = df[col].fillna(df[col].median())

# 5. Save the final processed signals
df.to_csv('data/processed_signals.csv', index=False)
print("Preprocessing complete. Signals saved.")
df[features].describe()